# Demand Forecasting — Aggregated Pipeline

This notebook studies the **aggregated demand-forecasting pipeline** embedded in the Aggregator Tool.

It covers:

1. Hourly aggregated next-24-hours forecasting.
2. 15-minute aggregated short-term forecasting.
3. Direct multi-output models versus recursive one-step models.
4. Optional weather-enhanced short-term forecasting.
5. Evaluation, plotting, artifact saving, and inference.

The service defines the aggregated forecasting endpoints under these FastAPI groups:

- `Aggregated Forecast Next 24 Hours`
- `Aggregated Short-Term Forecast (15')`


In [ ]:
# 1. Imports
import os
import io
import math
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Iterable, Optional, Sequence, Tuple, Dict, List

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

try:
    import asyncpg
except ImportError:
    asyncpg = None

try:
    from tensorflow.keras.models import Sequential, load_model
    from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout, Bidirectional
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.optimizers.schedules import ExponentialDecay
except ImportError:
    Sequential = load_model = GRU = LSTM = Dense = Dropout = Bidirectional = EarlyStopping = Adam = ExponentialDecay = None

pd.set_option('display.max_columns', 100)

## 1. Configuration

In [ ]:
@dataclass
class ForecastConfig:
    look_back_hourly: int = 48
    horizon_hourly: int = 24
    look_back_short: int = 16
    default_short_horizon: int = 4
    min_short_horizon: int = 4
    max_short_horizon: int = 20
    validation_fraction: float = 0.10
    random_seed: int = 42

CFG = ForecastConfig()
np.random.seed(CFG.random_seed)

In [ ]:
DB_SETTINGS = {
    'host': os.getenv('EHS_DB_HOST'),
    'port': int(os.getenv('EHS_DB_PORT', '5432')),
    'user': os.getenv('EHS_DB_USER'),
    'password': os.getenv('EHS_DB_PASSWORD'),
    'database': os.getenv('EHS_DB_NAME'),
}

ARTIFACT_DIR = os.getenv('FORECAST_ARTIFACT_DIR', './forecast_artifacts')
os.makedirs(ARTIFACT_DIR, exist_ok=True)

AGGREGATED_DEVICE_IDS: Optional[List[str]] = None

## 2. Data access

In [ ]:
def _validate_db_settings(settings: dict) -> None:
    missing = [k for k, v in settings.items() if v in (None, '')]
    if missing:
        raise RuntimeError(f'Missing database environment variables for: {missing}')
    if asyncpg is None:
        raise ImportError('Install asyncpg to fetch data from PostgreSQL: pip install asyncpg')


def _device_filter_sql(device_ids: Optional[Sequence[str]], start_param_index: int = 1) -> tuple[str, list]:
    if not device_ids:
        return '', []
    placeholders = ', '.join(f'${i}' for i in range(start_param_index, start_param_index + len(device_ids)))
    return f' AND device_id IN ({placeholders})', list(device_ids)


async def fetch_aggregated_hourly_data(device_ids: Optional[Sequence[str]] = None) -> pd.DataFrame:
    _validate_db_settings(DB_SETTINGS)
    device_sql, params = _device_filter_sql(device_ids)
    query = f'''
        SELECT hour, SUM(energy_usage) / 1000.0 AS energy_usage
        FROM hourly_data
        WHERE relay = 0 {device_sql}
        GROUP BY hour
        ORDER BY hour;
    '''
    conn = await asyncpg.connect(**DB_SETTINGS)
    try:
        rows = await conn.fetch(query, *params)
    finally:
        await conn.close()
    return pd.DataFrame(rows, columns=['hour', 'energy_usage'])


async def fetch_aggregated_quarterly_data(device_ids: Optional[Sequence[str]] = None) -> pd.DataFrame:
    _validate_db_settings(DB_SETTINGS)
    device_sql, params = _device_filter_sql(device_ids)
    query = f'''
        SELECT quarter, SUM(energy_usage) / 1000.0 AS energy_usage
        FROM quarterly_data
        WHERE relay = 0 {device_sql}
        GROUP BY quarter
        ORDER BY quarter;
    '''
    conn = await asyncpg.connect(**DB_SETTINGS)
    try:
        rows = await conn.fetch(query, *params)
    finally:
        await conn.close()
    return pd.DataFrame(rows, columns=['quarter', 'energy_usage'])

### Optional: load from CSV instead of database

- `hour, energy_usage`, or
- `quarter, energy_usage`.

In [ ]:
def load_series_csv(path: str, timestamp_col: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    expected = {timestamp_col, 'energy_usage'}
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f'Missing columns: {missing}')
    return df[[timestamp_col, 'energy_usage']].copy()

## 3. Feature engineering

In [ ]:
def add_hourly_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['energy_usage'] = pd.to_numeric(df['energy_usage'], errors='coerce')

    df['rolling_mean_6h'] = df['energy_usage'].rolling(window=6, min_periods=1).mean()
    df['rolling_std_6h'] = df['energy_usage'].rolling(window=6, min_periods=1).std()
    df['lag_1h'] = df['energy_usage'].shift(1)
    df['lag_3h'] = df['energy_usage'].shift(3)
    df['lag_6h'] = df['energy_usage'].shift(6)

    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    df['day'] = df.index.day
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    return df.dropna()


def add_quarterly_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['energy_usage'] = pd.to_numeric(df['energy_usage'], errors='coerce')

    df['rolling_mean_1h'] = df['energy_usage'].rolling(window=4, min_periods=1).mean()
    df['rolling_std_1h'] = df['energy_usage'].rolling(window=4, min_periods=1).std()
    df['lag_15m'] = df['energy_usage'].shift(1)
    df['lag_30m'] = df['energy_usage'].shift(2)
    df['lag_1h'] = df['energy_usage'].shift(4)
    df['lag_3h'] = df['energy_usage'].shift(12)
    df['lag_6h'] = df['energy_usage'].shift(24)

    df['minute'] = df.index.minute
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    df['day'] = df.index.day
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    return df.dropna()

In [ ]:
def prepare_time_index(df: pd.DataFrame, timestamp_col: str) -> pd.DataFrame:
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col])
    df = df.sort_values(timestamp_col).set_index(timestamp_col)
    df['energy_usage'] = pd.to_numeric(df['energy_usage'], errors='coerce')
    df['energy_usage'] = df['energy_usage'].ffill()
    return df

## 4. Scaling and sequence creation

In [ ]:
@dataclass
class PreprocessedData:
    df_scaled: pd.DataFrame
    target_scaler: MinMaxScaler
    feature_scaler: MinMaxScaler
    target_col: str
    feature_cols: list[str]
    model_cols: list[str]


def scale_for_forecasting(df_features: pd.DataFrame, fit: bool = True,
                          target_scaler: Optional[MinMaxScaler] = None,
                          feature_scaler: Optional[MinMaxScaler] = None,
                          feature_cols: Optional[list[str]] = None) -> PreprocessedData:
    target_col = 'energy_usage'
    if feature_cols is None:
        feature_cols = [c for c in df_features.columns if c != target_col]

    target_scaler = target_scaler or MinMaxScaler()
    feature_scaler = feature_scaler or MinMaxScaler()

    df_scaled = df_features.copy()
    if fit:
        df_scaled[[target_col]] = target_scaler.fit_transform(df_features[[target_col]])
        df_scaled[feature_cols] = feature_scaler.fit_transform(df_features[feature_cols])
    else:
        df_scaled[[target_col]] = target_scaler.transform(df_features[[target_col]])
        df_scaled[feature_cols] = feature_scaler.transform(df_features[feature_cols])

    model_cols = [target_col] + feature_cols
    return PreprocessedData(df_scaled[model_cols], target_scaler, feature_scaler, target_col, feature_cols, model_cols)


def create_direct_sequences(data: pd.DataFrame, look_back: int, horizon: int, target_col: str = 'energy_usage') -> tuple[np.ndarray, np.ndarray]:
    X, y = [], []
    for i in range(len(data) - look_back - horizon + 1):
        X.append(data.iloc[i:i + look_back].values)
        y.append(data.iloc[i + look_back:i + look_back + horizon][target_col].values)
    return np.asarray(X), np.asarray(y)


def create_recursive_sequences(data: pd.DataFrame, look_back: int, target_col: str = 'energy_usage') -> tuple[np.ndarray, np.ndarray]:
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data.iloc[i:i + look_back].values)
        y.append(data.iloc[i + look_back][target_col])
    return np.asarray(X), np.asarray(y).reshape(-1, 1)


def chronological_train_val_split(X: np.ndarray, y: np.ndarray, validation_fraction: float = 0.10):
    split_idx = int(len(X) * (1 - validation_fraction))
    return X[:split_idx], X[split_idx:], y[:split_idx], y[split_idx:]

## 5. Models

The service uses Bidirectional GRU for most aggregated models and Bidirectional LSTM in one short-term recursive variant.

For demand forecasting, the notebook compare two strategies:

1. **Direct multi-output:** one model predicts the full horizon at once.
2. **Recursive one-step:** one model predicts one step; its output is fed back repeatedly.

Direct models avoid error accumulation but are fixed to a trained horizon. Recursive models can forecast variable horizons but can drift.

In [ ]:
def build_direct_gru(input_shape: tuple[int, int], horizon: int, deep: bool = False):
    if Sequential is None:
        raise ImportError('Install tensorflow to build models.')
    if deep:
        lr_schedule = ExponentialDecay(initial_learning_rate=0.001, decay_steps=1000, decay_rate=0.9)
        optimizer = Adam(learning_rate=lr_schedule)
        layers = [
            Bidirectional(GRU(200, activation='tanh', return_sequences=True), input_shape=input_shape),
            Bidirectional(GRU(100, activation='tanh', return_sequences=True)),
            Bidirectional(GRU(100, activation='tanh', return_sequences=True)),
            Bidirectional(GRU(100, activation='tanh', return_sequences=False)),
            Dropout(0.3),
            Dense(horizon),
        ]
    else:
        optimizer = Adam(learning_rate=0.001)
        layers = [
            Bidirectional(GRU(200, activation='tanh', return_sequences=True), input_shape=input_shape),
            Bidirectional(GRU(100, activation='tanh', return_sequences=False)),
            Dropout(0.2),
            Dense(horizon),
        ]
    model = Sequential(layers)
    model.compile(optimizer=optimizer, loss='mean_squared_error')
    return model


def build_recursive_gru(input_shape: tuple[int, int]):
    if Sequential is None:
        raise ImportError('Install tensorflow to build models.')
    model = Sequential([
        Bidirectional(GRU(200, activation='tanh', return_sequences=True), input_shape=input_shape),
        Bidirectional(GRU(100, activation='tanh', return_sequences=True)),
        Bidirectional(GRU(100, activation='tanh', return_sequences=False)),
        Dropout(0.3),
        Dense(1),
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
    return model


def build_recursive_lstm(input_shape: tuple[int, int]):
    if Sequential is None:
        raise ImportError('Install tensorflow to build models.')
    model = Sequential([
        Bidirectional(LSTM(200, activation='tanh', return_sequences=True), input_shape=input_shape),
        Bidirectional(LSTM(100, activation='tanh', return_sequences=False)),
        Dropout(0.2),
        Dense(1),
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
    return model

## 6. Training helpers

In [ ]:
def inverse_target_2d(values: np.ndarray, scaler: MinMaxScaler) -> np.ndarray:
    original_shape = values.shape
    flat = values.reshape(-1, 1)
    inv = scaler.inverse_transform(flat).reshape(original_shape)
    return inv


def regression_metrics(y_true_scaled: np.ndarray, y_pred_scaled: np.ndarray, target_scaler: MinMaxScaler) -> dict:
    y_true = inverse_target_2d(y_true_scaled, target_scaler).flatten()
    y_pred = inverse_target_2d(y_pred_scaled, target_scaler).flatten()
    mean_usage = float(np.mean(y_true)) if len(y_true) else np.nan
    mae = mean_absolute_error(y_true, y_pred)
    return {
        'mse': float(mean_squared_error(y_true, y_pred)),
        'rmse': float(math.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mae),
        'mape_percent': float(mean_absolute_percentage_error(y_true, y_pred) * 100),
        'r2': float(r2_score(y_true, y_pred)),
        'mean_energy_usage': mean_usage,
        'relative_mae_percent': float((mae / mean_usage) * 100) if mean_usage else np.nan,
    }


def fit_model(model, X_train, y_train, X_val, y_val, epochs=50, batch_size=8, patience=5):
    early_stopping = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)
    history = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_val, y_val),
        callbacks=[early_stopping],
        verbose=1,
    )
    return history


def save_artifacts(prefix: str, model, prep: PreprocessedData, extra: Optional[dict] = None) -> None:
    model_path = os.path.join(ARTIFACT_DIR, f'{prefix}_model.keras')
    model.save(model_path)
    joblib.dump(prep.target_scaler, os.path.join(ARTIFACT_DIR, f'{prefix}_target_scaler.pkl'))
    joblib.dump(prep.feature_scaler, os.path.join(ARTIFACT_DIR, f'{prefix}_feature_scaler.pkl'))
    joblib.dump(prep.feature_cols, os.path.join(ARTIFACT_DIR, f'{prefix}_feature_cols.pkl'))
    if extra:
        joblib.dump(extra, os.path.join(ARTIFACT_DIR, f'{prefix}_extra.pkl'))
    print(f'Saved artifacts with prefix: {prefix}')

## 7. Pipeline A — hourly next-24-hours direct model

In [ ]:
async def train_hourly_direct_24h(df_hourly_raw: Optional[pd.DataFrame] = None):
    if df_hourly_raw is None:
        df_hourly_raw = await fetch_aggregated_hourly_data(AGGREGATED_DEVICE_IDS)

    if len(df_hourly_raw) < CFG.look_back_hourly + CFG.horizon_hourly:
        raise ValueError('Not enough hourly data for training.')

    df = prepare_time_index(df_hourly_raw, 'hour')
    df_features = add_hourly_features(df)
    prep = scale_for_forecasting(df_features, fit=True)

    X, y = create_direct_sequences(prep.df_scaled, CFG.look_back_hourly, CFG.horizon_hourly)
    X_train, X_val, y_train, y_val = chronological_train_val_split(X, y, CFG.validation_fraction)

    model = build_direct_gru((X.shape[1], X.shape[2]), horizon=CFG.horizon_hourly, deep=True)
    history = fit_model(model, X_train, y_train, X_val, y_val, epochs=50, batch_size=1, patience=10)

    y_pred = model.predict(X_val)
    metrics = regression_metrics(y_val, y_pred, prep.target_scaler)
    save_artifacts('aggregated_hourly_direct_24h', model, prep, {'metrics': metrics})
    return model, prep, history, metrics

In [ ]:
def forecast_hourly_direct_24h(model, prep: PreprocessedData, df_hourly_raw: pd.DataFrame) -> pd.DataFrame:
    df = prepare_time_index(df_hourly_raw, 'hour')
    df_features = add_hourly_features(df)
    df_scaled = scale_for_forecasting(
        df_features,
        fit=False,
        target_scaler=prep.target_scaler,
        feature_scaler=prep.feature_scaler,
        feature_cols=prep.feature_cols,
    ).df_scaled

    if len(df_scaled) < CFG.look_back_hourly:
        raise ValueError('Not enough hourly data for inference.')

    X_last = df_scaled.iloc[-CFG.look_back_hourly:].values.reshape(1, CFG.look_back_hourly, -1)
    pred_scaled = model.predict(X_last)
    pred = inverse_target_2d(pred_scaled, prep.target_scaler).flatten()

    timestamps = [df_scaled.index[-1] + pd.Timedelta(hours=i) for i in range(1, CFG.horizon_hourly + 1)]
    return pd.DataFrame({'timestamp': timestamps, 'forecasted_energy_usage': pred})

## 8. Pipeline B — aggregated 15-minute direct short-term model

In [ ]:
async def train_quarterly_direct_short(df_quarterly_raw: Optional[pd.DataFrame] = None, horizon: int = CFG.default_short_horizon):
    if not (CFG.min_short_horizon <= horizon <= CFG.max_short_horizon):
        raise ValueError(f'horizon must be between {CFG.min_short_horizon} and {CFG.max_short_horizon}')
    if df_quarterly_raw is None:
        df_quarterly_raw = await fetch_aggregated_quarterly_data(AGGREGATED_DEVICE_IDS)

    df = prepare_time_index(df_quarterly_raw, 'quarter')
    df_features = add_quarterly_features(df)
    prep = scale_for_forecasting(df_features, fit=True)

    X, y = create_direct_sequences(prep.df_scaled, CFG.look_back_short, horizon)
    X_train, X_val, y_train, y_val = chronological_train_val_split(X, y, CFG.validation_fraction)

    model = build_direct_gru((X.shape[1], X.shape[2]), horizon=horizon, deep=False)
    history = fit_model(model, X_train, y_train, X_val, y_val, epochs=200, batch_size=4, patience=5)

    y_pred = model.predict(X_val)
    metrics = regression_metrics(y_val, y_pred, prep.target_scaler)
    save_artifacts(f'aggregated_quarterly_direct_h{horizon}', model, prep, {'horizon': horizon, 'metrics': metrics})
    return model, prep, history, metrics

In [ ]:
def forecast_quarterly_direct_short(model, prep: PreprocessedData, df_quarterly_raw: pd.DataFrame, horizon: int) -> pd.DataFrame:
    df = prepare_time_index(df_quarterly_raw, 'quarter')
    df_features = add_quarterly_features(df)
    df_scaled = scale_for_forecasting(
        df_features,
        fit=False,
        target_scaler=prep.target_scaler,
        feature_scaler=prep.feature_scaler,
        feature_cols=prep.feature_cols,
    ).df_scaled

    X_last = df_scaled.iloc[-CFG.look_back_short:].values.reshape(1, CFG.look_back_short, -1)
    pred_scaled = model.predict(X_last)[:, :horizon]
    pred = inverse_target_2d(pred_scaled, prep.target_scaler).flatten()

    timestamps = [df_scaled.index[-1] + pd.Timedelta(minutes=15 * i) for i in range(1, horizon + 1)]
    return pd.DataFrame({'timestamp': timestamps, 'forecasted_energy_usage': pred})

## 9. Pipeline C — recursive forecasting

In [ ]:
def _recursive_forecast_recompute_features(
    model,
    prep: PreprocessedData,
    df_raw: pd.DataFrame,
    timestamp_col: str,
    add_feature_fn,
    look_back: int,
    horizon: int,
    step: pd.Timedelta,
) -> pd.DataFrame:
    history = prepare_time_index(df_raw, timestamp_col)[['energy_usage']].copy()
    predictions = []
    timestamps = []

    for i in range(horizon):
        df_features = add_feature_fn(history)
        df_scaled = scale_for_forecasting(
            df_features,
            fit=False,
            target_scaler=prep.target_scaler,
            feature_scaler=prep.feature_scaler,
            feature_cols=prep.feature_cols,
        ).df_scaled
        if len(df_scaled) < look_back:
            raise ValueError('Not enough data after feature engineering for recursive inference.')

        X_last = df_scaled.iloc[-look_back:].values.reshape(1, look_back, -1)
        pred_scaled = model.predict(X_last, verbose=0)[0, 0]
        pred = prep.target_scaler.inverse_transform([[pred_scaled]])[0, 0]

        next_ts = history.index[-1] + step
        history.loc[next_ts, 'energy_usage'] = pred
        timestamps.append(next_ts)
        predictions.append(pred)

    return pd.DataFrame({'timestamp': timestamps, 'forecasted_energy_usage': predictions})


async def train_hourly_recursive_24h(df_hourly_raw: Optional[pd.DataFrame] = None):
    if df_hourly_raw is None:
        df_hourly_raw = await fetch_aggregated_hourly_data(AGGREGATED_DEVICE_IDS)
    df = prepare_time_index(df_hourly_raw, 'hour')
    df_features = add_hourly_features(df)
    prep = scale_for_forecasting(df_features, fit=True)
    X, y = create_recursive_sequences(prep.df_scaled, CFG.look_back_hourly)
    X_train, X_val, y_train, y_val = chronological_train_val_split(X, y, CFG.validation_fraction)

    model = build_recursive_gru((X.shape[1], X.shape[2]))
    history = fit_model(model, X_train, y_train, X_val, y_val, epochs=50, batch_size=1, patience=10)
    y_pred = model.predict(X_val)
    metrics = regression_metrics(y_val, y_pred, prep.target_scaler)
    save_artifacts('aggregated_hourly_recursive_24h', model, prep, {'metrics': metrics})
    return model, prep, history, metrics


def forecast_hourly_recursive_24h(model, prep: PreprocessedData, df_hourly_raw: pd.DataFrame) -> pd.DataFrame:
    return _recursive_forecast_recompute_features(
        model, prep, df_hourly_raw, 'hour', add_hourly_features,
        CFG.look_back_hourly, CFG.horizon_hourly, pd.Timedelta(hours=1)
    )


async def train_quarterly_recursive_short(df_quarterly_raw: Optional[pd.DataFrame] = None):
    if df_quarterly_raw is None:
        df_quarterly_raw = await fetch_aggregated_quarterly_data(AGGREGATED_DEVICE_IDS)
    df = prepare_time_index(df_quarterly_raw, 'quarter')
    df_features = add_quarterly_features(df)
    prep = scale_for_forecasting(df_features, fit=True)
    X, y = create_recursive_sequences(prep.df_scaled, CFG.look_back_short)
    X_train, X_val, y_train, y_val = chronological_train_val_split(X, y, CFG.validation_fraction)

    model = build_recursive_lstm((X.shape[1], X.shape[2]))
    history = fit_model(model, X_train, y_train, X_val, y_val, epochs=50, batch_size=8, patience=5)
    y_pred = model.predict(X_val)
    metrics = regression_metrics(y_val, y_pred, prep.target_scaler)
    save_artifacts('aggregated_quarterly_recursive_short', model, prep, {'metrics': metrics})
    return model, prep, history, metrics


def forecast_quarterly_recursive_short(model, prep: PreprocessedData, df_quarterly_raw: pd.DataFrame, horizon: int = CFG.default_short_horizon) -> pd.DataFrame:
    return _recursive_forecast_recompute_features(
        model, prep, df_quarterly_raw, 'quarter', add_quarterly_features,
        CFG.look_back_short, horizon, pd.Timedelta(minutes=15)
    )

## 10. Optional weather-enhanced aggregated short-term model

In [ ]:
WEATHER_FEATURES = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'cloud_cover', 'precipitation']


def merge_weather_asof(energy_df: pd.DataFrame, weather_df: pd.DataFrame, timestamp_col: str = 'quarter') -> pd.DataFrame:
    energy = energy_df.copy()
    weather = weather_df.copy()
    energy[timestamp_col] = pd.to_datetime(energy[timestamp_col]).dt.tz_localize(None)
    weather['timestamp'] = pd.to_datetime(weather['timestamp']).dt.tz_localize(None)

    merged = pd.merge_asof(
        energy.sort_values(timestamp_col),
        weather[['timestamp'] + WEATHER_FEATURES].sort_values('timestamp'),
        left_on=timestamp_col,
        right_on='timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('30min'),
    ).drop(columns=['timestamp'])
    return merged.dropna(subset=WEATHER_FEATURES)


def add_quarterly_features_with_weather(df: pd.DataFrame) -> pd.DataFrame:
    base = add_quarterly_features(df)
    for col in WEATHER_FEATURES:
        if col not in base.columns:
            raise ValueError(f'Missing weather feature: {col}')
    return base

## 11. Evaluation and visualization

In [ ]:
def plot_forecast(forecast_df: pd.DataFrame, actual_df: Optional[pd.DataFrame] = None, title: str = 'Forecast'):
    plt.figure(figsize=(12, 5))
    if actual_df is not None:
        actual = actual_df.copy()
        ts_col = 'timestamp' if 'timestamp' in actual.columns else actual.columns[0]
        plt.plot(pd.to_datetime(actual[ts_col]), actual['energy_usage'], label='Actual')
    plt.plot(pd.to_datetime(forecast_df['timestamp']), forecast_df['forecasted_energy_usage'], marker='o', label='Forecast')
    plt.title(title)
    plt.xlabel('Time')
    plt.ylabel('Energy usage (kWh)')
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_training_history(history):
    plt.figure(figsize=(10, 4))
    plt.plot(history.history['loss'], label='Training loss')
    if 'val_loss' in history.history:
        plt.plot(history.history['val_loss'], label='Validation loss')
    plt.title('Training history')
    plt.xlabel('Epoch')
    plt.ylabel('MSE loss')
    plt.legend()
    plt.grid(True)
    plt.show()